In [1]:
%pip install numpy
%pip install scipy
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import math
import numpy as np
import numpy.linalg as linalg
from scipy.integrate import ode
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
import ipywidgets as widgets
from IPython.display import display

In [66]:
g = 9.8
EPSILON = 1e-3

def compute_gravity_proj(m: float, surface_normal: np.ndarray) -> np.ndarray:
    basis1 = np.array([1, 0, 0])
    basis2 = np.cross(basis1, surface_normal)

    full_gravity = np.array([0, -m * g, 0])
    return (linalg.inv(np.array([basis1, basis2, surface_normal])) @ full_gravity)[:2]

def compute_friction(v: np.ndarray, w: np.ndarray, m: float, r: float, I: np.ndarray, 
                     friction_coefficient: float, gravity: np.ndarray, 
                     surface_normal: np.ndarray) -> np.ndarray:
    v_contact = v - w * r

    max_friction = -np.dot(friction_coefficient * np.array([0, -m * g, 0]), surface_normal)

    stable_friction = gravity / (I[:2] / m / r ** 2 + 1)
    if linalg.norm(stable_friction) > max_friction:
        stable_friction = gravity / linalg.norm(gravity)

    if linalg.norm(v_contact) > EPSILON:
        return max_friction * v_contact / linalg.norm(v_contact)
    else:
        return stable_friction

def make_integrator(m: float, r: float, I: np.ndarray, friction_coefficient: float,
                    get_surface_normal, x0: np.ndarray, v0: np.ndarray, w0: np.ndarray) -> ode:
    def derivatives(_, state: np.ndarray) -> np.ndarray:
        v = state[2:4]
        w = state[4:]

        surface_normal = get_surface_normal()
        gravity = compute_gravity_proj(m, surface_normal)
        friction = compute_friction(v, w, m, r, I, friction_coefficient, gravity, surface_normal)

        derivatives = np.array([
            *v,
            *(gravity - friction),
            *(friction * r / I[:2])
        ])
        return derivatives
    
    integrator = ode(derivatives)
    integrator.set_initial_value((*x0, *v0, *w0))
    return integrator


In [50]:
%pip install ipycanvas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import ipycanvas as ipyc
from ipycanvas import hold_canvas
import numpy as np
from IPython.display import display
import time
import asyncio
import traceback

canvas = ipyc.Canvas(width=600, height=400)

circle_x, circle_y = 300, 200
surface_normal = np.array([0, 0, 1])

M = 1
R = 1
integrator = make_integrator(M, R, np.repeat(2 / 5 * R * M ** 2, 3), 1.0, 
                             lambda: surface_normal / linalg.norm(surface_normal),
                             np.array([circle_x, circle_y]), np.zeros(2), np.zeros(2))

circle_radius = 30
is_dragging = False
last_mouse_pos = None

start_time = time.time()

def handle_mouse_down(x, y):
    global is_dragging 
    is_dragging = True

def handle_mouse_move(x, y):
    global surface_normal
    surface_normal[0] = x
    surface_normal[1] = y

def handle_mouse_up(x, y):
    global is_dragging
    is_dragging = False

canvas.on_mouse_down(handle_mouse_down)
canvas.on_mouse_move(handle_mouse_move)
canvas.on_mouse_up(handle_mouse_up)

frame = 0

def update():
    x, y, _, _, _, _ = integrator.integrate(time.time() - start_time)
    
    global circle_x, circle_y
    circle_x = x
    circle_y = y

    global frame
    frame += 1
    if frame % 10 == 0:
        print(surface_normal)

def draw():
    with hold_canvas():
        canvas.clear()
        canvas.fill_style = 'red'
        canvas.fill_circle(circle_x, circle_y, circle_radius)

async def game_loop():
    try:
        while True:
            update()
            draw()
            await asyncio.sleep(1/60)  # 60 FPS
    except Exception as e:
        print(e)
        traceback.print_exc()

display(canvas)
asyncio.create_task(game_loop())

Canvas(height=400, width=600)

<Task pending name='Task-155406' coro=<game_loop() running at C:\Users\PJutch\AppData\Local\Temp\ipykernel_86800\1013016012.py:66>>

[349 207   1]
[232 222   1]
[292 304   1]
[312 140   1]
[322  50   1]
[371 242   1]
[393 189   1]
[228 153   1]
[321 265   1]
[253 209   1]
[363 359   1]
[363 359   1]
[393 363   1]
[412 378   1]
[226  77   1]
[381 167   1]
[387  95   1]
[367  83   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  85   1]
[367  